# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [1]:
# ========================================================================
# EXPERIMENT 1: CONTROL (Original Images)
# FIX: Using EfficientNetV2B2 and IMAGE_SIZE=260 to fit in memory
# ========================================================================
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, GlobalAveragePooling2D, Dropout, Dense)
# --- FIX: Using a slightly smaller, but still very powerful, model ---
from tensorflow.keras.applications import EfficientNetV2B2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.optimizers.schedules import CosineDecayRestarts
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# Enable mixed precision for faster training
tf.keras.mixed_precision.set_global_policy('mixed_float16')
print("Mixed precision enabled:", tf.keras.mixed_precision.global_policy().name)

Mixed precision enabled: mixed_float16


### overall configuration and Path set up

In [ ]:
# ========================================================================
# SECTION 1: CONFIGURATION
# ========================================================================
BASE_PATH = 'D:\downloads\skin cancer dataset\skin cancer dataset'
CSV_PATH = os.path.join(BASE_PATH, 'HAM10000_metadata.csv')

# --- FIX: Adjusted Image Size and Batch Size for B2 Model ---
IMAGE_SIZE = 260 # Recommended size for EfficientNetV2B2
BATCH_SIZE = 16  # We can try 16 again as B2 is smaller than B3

FEATURE_EXTRACTION_EPOCHS = 10
FINE_TUNING_EPOCHS = 40
LEARNING_RATE_HEAD = 1e-3
LEARNING_RATE_FINE_TUNE_INITIAL = 1e-5
LABEL_SMOOTHING = 0.1
AUTOTUNE = tf.data.AUTOTUNE

In [ ]:
# ========================================================================
# SECTION 2: DATA LOADING AND PREPARATION
# ========================================================================
print("\n--- Loading and preparing data (CONTROL - ORIGINAL IMAGES) ---")
df = pd.read_csv(CSV_PATH)
df['path'] = df['image_id'].apply(lambda x: os.path.join(BASE_PATH, 'HAM10000_images_part_1' if os.path.exists(os.path.join(BASE_PATH, 'HAM10000_images_part_1', x + '.jpg')) else 'HAM10000_images_part_2', x + '.jpg'))
df = df.dropna(subset=['path'])
label_columns = sorted(df['dx'].unique())
df = pd.concat([df, pd.get_dummies(df['dx'], dtype='float32')], axis=1)

train_df, validation_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['dx'])
print(f"\nTraining set size: {len(train_df)}, Validation set size: {len(validation_df)}")


In [ ]:
# ========================================================================
# SECTION 3: tf.data PIPELINE WITH CUTMIX
# ========================================================================
print(f"\n--- Building tf.data pipelines with CutMix (IMAGE_SIZE={IMAGE_SIZE})...")

def parse_image(filepath, label):
    image = tf.io.read_file(filepath)
    image = tf.image.decode_jpeg(image, channels=3)
    # FIX: Changed resize method from BICUBIC to BILINEAR for faster preprocessing
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE], method=tf.image.ResizeMethod.BILINEAR)
    return image, label

def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, lower=0.9, upper=1.1)
    return image, label

def cutmix(image1, label1, image2, label2, alpha=1.0):
    lambda_ = tf.random.uniform(shape=[], minval=0., maxval=1.)
    h, w = IMAGE_SIZE, IMAGE_SIZE
    cut_ratio = tf.math.sqrt(1.0 - lambda_)
    cy = tf.random.uniform(shape=[], minval=0, maxval=h, dtype=tf.int32)
    cx = tf.random.uniform(shape=[], minval=0, maxval=w, dtype=tf.int32)
    cut_h = tf.cast(tf.cast(h, tf.float32) * cut_ratio, tf.int32)
    cut_w = tf.cast(tf.cast(w, tf.float32) * cut_ratio, tf.int32)
    y1 = tf.clip_by_value(cy - cut_h // 2, 0, h); y2 = tf.clip_by_value(cy + cut_h // 2, 0, h)
    x1 = tf.clip_by_value(cx - cut_w // 2, 0, w); x2 = tf.clip_by_value(cx + cut_w // 2, 0, w)
    mask = tf.pad(tf.ones_like(image1[y1:y2, x1:x2, :]), [[y1, h - y2], [x1, w - x2], [0, 0]])
    mixed_image = (1.0 - mask) * image1 + mask * image2
    mix_ratio = tf.cast((x2 - x1) * (y2 - y1), tf.float32) / tf.cast(h * w, tf.float32)
    mixed_label = (1.0 - mix_ratio) * label1 + mix_ratio * label2
    return mixed_image, mixed_label

def build_dataset(df, is_training=True):
    ds = tf.data.Dataset.from_tensor_slices((df['path'].values, df[label_columns].values))
    if is_training:
        ds = ds.shuffle(buffer_size=len(df))
    ds = ds.map(parse_image, num_parallel_calls=AUTOTUNE)
    if is_training:
        ds2 = ds.shuffle(buffer_size=len(df))
        ds_zipped = tf.data.Dataset.zip((ds, ds2))
        ds = ds_zipped.map(lambda ds_one, ds_two: cutmix(ds_one[0], ds_one[1], ds_two[0], ds_two[1]),
                           num_parallel_calls=AUTOTUNE)
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(buffer_size=AUTOTUNE)
    return ds

train_dataset = build_dataset(train_df)
validation_dataset = build_dataset(validation_df, is_training=False)

In [ ]:
# ========================================================================
# SECTION 4: BUILDING THE ULTIMATE TRANSFER LEARNING MODEL
# ========================================================================
print(f"\n--- Building transfer learning model with EfficientNetV2B2...")
# --- FIX: Changed model to B2 ---
base_model = EfficientNetV2B2(
    include_top=False,
    weights='imagenet',
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)
)
base_model.trainable = False
inputs = Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.4)(x)
outputs = Dense(len(label_columns), activation='softmax', dtype='float32')(x)
model = Model(inputs, outputs)
model.summary()

In [ ]:
# ========================================================================
# SECTION 5: STAGE 1 - TRAINING THE CLASSIFICATION HEAD
# ========================================================================
print(f"\n--- STAGE 1: Training the head...")
model.compile(optimizer=AdamW(learning_rate=LEARNING_RATE_HEAD),
              loss=CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
              metrics=['accuracy'])
checkpoint_head = ModelCheckpoint('best_head_model_control.keras', monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)
history_head = model.fit(train_dataset, epochs=FEATURE_EXTRACTION_EPOCHS,
                         validation_data=validation_dataset, callbacks=[checkpoint_head])
model.load_weights('best_head_model_control.keras')

In [ ]:
# ========================================================================
# SECTION 6: STAGE 2 - FINE-TUNING THE ENTIRE MODEL
# ========================================================================
print(f"\n--- STAGE 2: Fine-tuning the entire model...")

# Load the best head model weights from the saved file
# This assumes the model architecture has already been built in SECTION 4
try:
    model.load_weights('best_head_model_control.keras')
    print("Successfully loaded best head model weights.")
except Exception as e:
    print(f"Error loading best head model weights: {e}")
    print("Proceeding with current model state. Ensure the head was trained or weights are available.")

base_model.trainable = True # Make the base model trainable

# Recompile the model after making base_model trainable
steps_per_epoch_ft = len(train_df) // BATCH_SIZE
lr_schedule_ft = CosineDecayRestarts(initial_learning_rate=LEARNING_RATE_FINE_TUNE_INITIAL,
                                     first_decay_steps=steps_per_epoch_ft * 10, t_mul=2.0, m_mul=0.9)
optimizer_ft = AdamW(learning_rate=lr_schedule_ft, weight_decay=1e-5)
model.compile(optimizer=optimizer_ft,
              loss=CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
              metrics=['accuracy'])

checkpoint_ft = ModelCheckpoint('best_fine_tuned_model_control.keras', monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)
early_stop_ft = EarlyStopping(monitor='val_loss', patience=10, verbose=1)

# Set initial_epoch to FEATURE_EXTRACTION_EPOCHS as we are conceptually starting fine-tuning
# after the head training phase, which is now loaded from a file.
history_fine_tune = model.fit(train_dataset,
                            epochs=FEATURE_EXTRACTION_EPOCHS + FINE_TUNING_EPOCHS,
                            initial_epoch=FEATURE_EXTRACTION_EPOCHS, # Start from where head training would have left off
                            validation_data=validation_dataset,
                            callbacks=[early_stop_ft, checkpoint_ft])

model.load_weights('best_fine_tuned_model_control.keras')

In [ ]:
# ========================================================================
# SECTION 7: VISUALIZATION & TTA EVALUATION
# ========================================================================
def plot_history(history_head, history_fine_tune):
    acc = history_head.history['accuracy'] + history_fine_tune.history['accuracy']
    val_acc = history_head.history['val_accuracy'] + history_fine_tune.history['val_accuracy']
    loss = history_head.history['loss'] + history_fine_tune.history['loss']
    val_loss = history_head.history['val_loss'] + history_fine_tune.history['val_loss']
    epochs_range = range(len(acc))
    plt.figure(figsize=(14, 6))
    plt.subplot(1, 2, 1); plt.plot(epochs_range, acc, label='Training Accuracy'); plt.plot(epochs_range, val_acc, label='Validation Accuracy')
    plt.axvline(x=len(history_head.epoch)-1, color='r', linestyle='--', label='Fine-tuning Start')
    plt.legend(loc='lower right'); plt.title('Training and Validation Accuracy'); plt.xlabel('Epoch'); plt.ylabel('Accuracy')
    plt.subplot(1, 2, 2); plt.plot(epochs_range, loss, label='Training Loss'); plt.plot(epochs_range, val_loss, label='Validation Loss')
    plt.axvline(x=len(history_head.epoch)-1, color='r', linestyle='--', label='Fine-tuning Start')
    plt.legend(loc='upper right'); plt.title('Training and Validation Loss'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.show()

print("\n--- Plotting combined training history...")
plot_history(history_head, history_fine_tune)

def predict_with_tta(model, image_path, target_size):
    img = load_img(image_path, target_size=target_size); img_array = img_to_array(img)
    img_array_expanded = tf.expand_dims(img_array, axis=0)
    flipped_img_array = tf.image.flip_left_right(img_array_expanded)
    pred_original = model.predict(img_array_expanded, verbose=0)
    pred_flipped = model.predict(flipped_img_array, verbose=0)
    return np.mean([pred_original, pred_flipped], axis=0)[0]

print("\n--- Evaluating final model with Test-Time Augmentation (TTA)...")
correct_tta_predictions = 0; true_labels = validation_df[label_columns].values
for i, (_, row) in enumerate(tqdm(validation_df.iterrows(), total=len(validation_df), desc="TTA Progress")):
    tta_pred = predict_with_tta(model, row['path'], (IMAGE_SIZE, IMAGE_SIZE))
    if np.argmax(tta_pred) == np.argmax(true_labels[i]): correct_tta_predictions += 1

tta_accuracy = correct_tta_predictions / len(validation_df)
final_val_accuracy = max(history_fine_tune.history.get('val_accuracy', [0]))
print(f"\n--- Final Results (CONTROL) ---\nBest Validation Accuracy (from training): {final_val_accuracy:.4f}\nValidation Accuracy with TTA: {tta_accuracy:.4f}")